In [1]:
import pandas as pd 
import numpy as np 

In [2]:
order_level = pd.read_csv("/home/user/Suraj/Work/E_Commerce/Cleaned/ready/order_level.csv")
customer_level = pd.read_csv("/home/user/Suraj/Work/E_Commerce/Cleaned/ready/customer_level.csv")
seller_level = pd.read_csv("/home/user/Suraj/Work/E_Commerce/Cleaned/ready/seller_level.csv")

### Revenue Leakage & LTV Retention

In [3]:
order_level

,Unnamed: 0,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_time_days,...,is_delayed,order_value,total_freight_value,item_count,total_payment_value,payment_installments,payment_type,review_score,review_creation_date,customer_unique_id
0,0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,8.0,...,False,29.99,8.72,1.0,38.71,1.0,voucher,4.0,2017-10-11 00:00:00,7c396fd4830fd04220f754e42b4e5bff
1,1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,13.0,...,False,118.70,22.76,1.0,141.46,1.0,boleto,4.0,2018-08-08 00:00:00,af07308b275d755c9edb36a90c618231
2,2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,9.0,...,False,159.90,19.22,1.0,179.12,3.0,credit_card,5.0,2018-08-18 00:00:00,3a653a41f6f9fc3d2a113cf8398680e8
3,3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,13.0,...,False,45.00,27.20,1.0,72.20,1.0,credit_card,5.0,2017-12-03 00:00:00,7c142cf63193a1473d2e66489a9ae977
4,4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,2.0,...,False,19.90,8.72,1.0,28.62,1.0,credit_card,5.0,2018-02-17 00:00:00,72632f0f9dd73dfee390c9b22eb56dd6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99436,99436,9c5dedf39a927c1b2549525ed64a053c,39bd1228ee8140590ac3aca26f2dfe00,delivered,2017-03-09 09:54:05,2017-03-09 09:54:05,2017-03-10 11:18:03,2017-03-17 15:08:01,2017-03-28,8.0,...,False,72.00,13.08,1.0,85.08,3.0,credit_card,5.0,2017-03-22 00:00:00,6359f309b166b0196dbf7ad2ac62bb5a
99437,99437,63943bddc261676b46f01ca7ac2f7bd8,1fca14ff2861355f6e5f14306ff977a7,delivered,2018-02-06 12:58:58,2018-02-06 13:10:37,2018-02-07 23:22:42,2018-02-28 17:37:56,2018-03-02,22.0,...,False,174.90,20.10,1.0,195.00,3.0,credit_card,4.0,2018-03-01 00:00:00,da62f9e57a76d978d02ab5362c509660
99438,99438,83c1379a015df1e13d02aae0204711ab,1aa71eb042121263aafbe80c1b562c9c,delivered,2017-08-27 14:46:43,2017-08-27 15:04:16,2017-08-28 20:52:26,2017-09-21 11:24:17,2017-09-27,24.0,...,False,205.99,65.02,1.0,271.01,5.0,credit_card,5.0,2017-09-22 00:00:00,737520a9aad80b3fbbdad19b66b37b30
99439,99439,11c177c8e97725db2631073c19f07b62,b331b74b18dc79bcdf6532d51e1637c1,delivered,2018-01-08 21:28:27,2018-01-08 21:36:21,2018-01-12 15:35:03,2018-01-25 23:32:54,2018-02-15,17.0,...,False,359.98,81.18,2.0,441.16,4.0,credit_card,2.0,2018-01-26 00:00:00,5097a5312c8b157bb7be58ae360ef43c


In [4]:
order_level['min_timestamp'] = order_level.groupby('customer_unique_id')['order_purchase_timestamp'].transform('min')

In [5]:
order_level['is_first_order'] = order_level['order_purchase_timestamp'] == order_level['min_timestamp']

In [6]:
order_level.columns

Index(['Unnamed: 0', 'order_id', 'customer_id', 'order_status',
       'order_purchase_timestamp', 'order_approved_at',
       'order_delivered_carrier_date', 'order_delivered_customer_date',
       'order_estimated_delivery_date', 'delivery_time_days',
       'delivery_delay_days', 'is_delayed', 'order_value',
       'total_freight_value', 'item_count', 'total_payment_value',
       'payment_installments', 'payment_type', 'review_score',
       'review_creation_date', 'customer_unique_id', 'min_timestamp',
       'is_first_order'],
      dtype='object')

In [7]:
del order_level['min_timestamp']

In [8]:
order_level[order_level['is_first_order']==True]

,Unnamed: 0,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_time_days,...,order_value,total_freight_value,item_count,total_payment_value,payment_installments,payment_type,review_score,review_creation_date,customer_unique_id,is_first_order
1,1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,13.0,...,118.70,22.76,1.0,141.46,1.0,boleto,4.0,2018-08-08 00:00:00,af07308b275d755c9edb36a90c618231,True
2,2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,9.0,...,159.90,19.22,1.0,179.12,3.0,credit_card,5.0,2018-08-18 00:00:00,3a653a41f6f9fc3d2a113cf8398680e8,True
3,3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,13.0,...,45.00,27.20,1.0,72.20,1.0,credit_card,5.0,2017-12-03 00:00:00,7c142cf63193a1473d2e66489a9ae977,True
4,4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,2.0,...,19.90,8.72,1.0,28.62,1.0,credit_card,5.0,2018-02-17 00:00:00,72632f0f9dd73dfee390c9b22eb56dd6,True
5,5,a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,delivered,2017-07-09 21:57:05,2017-07-09 22:10:13,2017-07-11 14:58:04,2017-07-26 10:57:55,2017-08-01,16.0,...,147.90,27.36,1.0,175.26,6.0,credit_card,4.0,2017-07-27 00:00:00,80bb27c7c16e8f973207a5086ab329e2,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99436,99436,9c5dedf39a927c1b2549525ed64a053c,39bd1228ee8140590ac3aca26f2dfe00,delivered,2017-03-09 09:54:05,2017-03-09 09:54:05,2017-03-10 11:18:03,2017-03-17 15:08:01,2017-03-28,8.0,...,72.00,13.08,1.0,85.08,3.0,credit_card,5.0,2017-03-22 00:00:00,6359f309b166b0196dbf7ad2ac62bb5a,True
99437,99437,63943bddc261676b46f01ca7ac2f7bd8,1fca14ff2861355f6e5f14306ff977a7,delivered,2018-02-06 12:58:58,2018-02-06 13:10:37,2018-02-07 23:22:42,2018-02-28 17:37:56,2018-03-02,22.0,...,174.90,20.10,1.0,195.00,3.0,credit_card,4.0,2018-03-01 00:00:00,da62f9e57a76d978d02ab5362c509660,True
99438,99438,83c1379a015df1e13d02aae0204711ab,1aa71eb042121263aafbe80c1b562c9c,delivered,2017-08-27 14:46:43,2017-08-27 15:04:16,2017-08-28 20:52:26,2017-09-21 11:24:17,2017-09-27,24.0,...,205.99,65.02,1.0,271.01,5.0,credit_card,5.0,2017-09-22 00:00:00,737520a9aad80b3fbbdad19b66b37b30,True
99439,99439,11c177c8e97725db2631073c19f07b62,b331b74b18dc79bcdf6532d51e1637c1,delivered,2018-01-08 21:28:27,2018-01-08 21:36:21,2018-01-12 15:35:03,2018-01-25 23:32:54,2018-02-15,17.0,...,359.98,81.18,2.0,441.16,4.0,credit_card,2.0,2018-01-26 00:00:00,5097a5312c8b157bb7be58ae360ef43c,True


In [9]:
first_orders = order_level[order_level['is_first_order'] == True].copy()

In [10]:
customer_level_demo = customer_level.copy()

In [11]:
merged_data = customer_level_demo.merge(first_orders[['customer_unique_id', 'is_delayed']], 
                                        on='customer_unique_id', 
                                        how='inner')

In [12]:
merged_data['is_repeat'] = merged_data['total_orders']>1

In [13]:
result =  merged_data.groupby('is_delayed').agg(
    total_customers=('customer_unique_id', 'count'),
    repeat_rate=('is_repeat', lambda x: x.mean() * 100)
).reset_index()
result

,is_delayed,total_customers,repeat_rate
0,False,90012,3.440652
1,True,6362,2.797862


In [15]:
# Average LTV of repeat customers
repeat_customers = customer_level[customer_level['total_orders'] > 1]
avg_ltv_repeat = repeat_customers['total_revenue'].mean()
print(f"Avg LTV (repeat): ${avg_ltv_repeat:,.2f}")

# Lost repeat customers
delayed_first_order_count = 6362  # Your number from A
retention_gap = 0.0065  # 0.65% as decimal
lost_repeat = delayed_first_order_count * retention_gap
print(f"Lost repeat customers: {lost_repeat:,.0f}")

# Revenue at risk
revenue_at_risk = lost_repeat * avg_ltv_repeat
print(f"Revenue at Risk: ${revenue_at_risk:,.2f}")

Avg LTV (repeat): $277.30
Lost repeat customers: 41
Revenue at Risk: $11,467.03


In [17]:
# FINAL INSIGHT 
# Total customers
total_customers = customer_level['customer_unique_id'].nunique()

# Current repeat rate
current_repeat_rate = 0.031  # 3.1%

# Target repeat rate (conservative)
target_repeat_rate = 0.15  # 15%

# Current repeat customers
current_repeat = int(total_customers * current_repeat_rate)

# Target repeat customers
target_repeat = int(total_customers * target_repeat_rate)

# Gap
repeat_gap = target_repeat - current_repeat

# Avg LTV
avg_ltv = 277.30

# Opportunity
revenue_opportunity = repeat_gap * avg_ltv

print(f"Total Customers: {total_customers:,}")
print(f"Current Repeat Rate: {current_repeat_rate:.1%}")
print(f"Current Repeat Customers: {current_repeat:,}")
print(f"Target Repeat Customers (15%): {target_repeat:,}")
print(f"Retention Gap: {repeat_gap:,}")
print(f"Avg Repeat Customer LTV: ${avg_ltv:,.2f}")
print(f"Revenue Opportunity: ${revenue_opportunity:,.2f}")

Total Customers: 96,096
Current Repeat Rate: 3.1%
Current Repeat Customers: 2,978
Target Repeat Customers (15%): 14,414
Retention Gap: 11,436
Avg Repeat Customer LTV: $277.30
Revenue Opportunity: $3,171,202.80
